In [1]:
import random

locations = {
    "Restaurant": (0, 0),
    "Customer1": (2, 3),
    "Customer2": (5, 4),
    "Customer3": (6, 1),
    "Customer4": (3, -2),
    "Customer5": (1, -3)
}

NUM_ANTS = 10
ITERATIONS = 100
ALPHA = 1.0
BETA = 2.0
RHO = 0.5
Q = 100

cities = list(locations.keys())
n = len(cities)

distance = [[0.0] * n for _ in range(n)]
pheromone = [[0.1] * n for _ in range(n)]

for i in range(n):
    for j in range(n):
        if i != j:
            x1, y1 = locations[cities[i]]
            x2, y2 = locations[cities[j]]
            distance[i][j] = (
                (x2 - x1) ** 2 + (y2 - y1) ** 2
            ) ** 0.5
        else:
            pheromone[i][j] = 0.0


def route_distance(route):
    total = 0

    for i in range(len(route) - 1):
        total += distance[route[i]][route[i + 1]]

    return total


def select_next_city(current, unvisited):
    weights = []

    for city in unvisited:
        tau = pheromone[current][city] ** ALPHA
        eta = (1 / distance[current][city]) ** BETA
        weights.append(tau * eta)

    return random.choices(
        unvisited,
        weights=weights,
        k=1
    )[0]


def construct_route():
    start = cities.index("Restaurant")
    route = [start]
    unvisited = [i for i in range(n) if i != start]
    current = start

    while unvisited:
        next_city = select_next_city(current, unvisited)
        route.append(next_city)
        unvisited.remove(next_city)
        current = next_city

    route.append(start)
    return route


def update_pheromones(routes, lengths):
    for i in range(n):
        for j in range(n):
            pheromone[i][j] *= (1 - RHO)

    for route, length in zip(routes, lengths):
        deposit = Q / length

        for i in range(len(route) - 1):
            u = route[i]
            v = route[i + 1]

            pheromone[u][v] += deposit
            pheromone[v][u] += deposit


def optimize_delivery_route():
    best_route = None
    best_distance = float("inf")

    for iteration in range(ITERATIONS):
        routes = []
        lengths = []

        for ant in range(NUM_ANTS):
            route = construct_route()
            length = route_distance(route)

            routes.append(route)
            lengths.append(length)

            if length < best_distance:
                best_distance = length
                best_route = route[:]

        update_pheromones(routes, lengths)

    return best_route, best_distance


best_route, best_distance = optimize_delivery_route()

print("OPTIMIZED FOOD DELIVERY ROUTE")
print("--------------------------------")

for i, city_index in enumerate(best_route):
    print(f"{i + 1}. {cities[city_index]}")

print(f"\nTotal route distance: {best_distance:.2f} units")

OPTIMIZED FOOD DELIVERY ROUTE
--------------------------------
1. Restaurant
2. Customer1
3. Customer2
4. Customer3
5. Customer4
6. Customer5
7. Restaurant

Total route distance: 19.57 units
